# Try ScreamingFace against the hosted engine

ScreamingFace lets you describe a sample as a Model, combine several into a Fusion, and score
either one against a benchmark. This notebook runs that whole path against our hosted engine, so
you do not need to start anything locally.

It has two parts:

- **Part 1** reproduces the DRACO comparison: define a few Models, combine them into a Fusion, run
  them against the `draco/lite` benchmark, and read the Report.
- **Part 2** takes the URL4 address of the run you just did, turns it back into editable Python,
  changes one model, and runs it again.

Everything that spends money is behind an explicit switch, so running every cell top to bottom
makes no paid calls until you set one.

## Before you start

- Install the SDK and notebook/runtime extras: `pip install "screamingface[runtime,notebook]"`.
- Have an OpenRouter API key ready. You enter it once in the connection panel below; the notebook
  never stores it, and it goes to the hosted engine for validation and encrypted storage.
- You need access to the hosted engine. The connection panel opens a Cloudflare Access login the
  first time you connect.

Retrieval for DRACO (web search and fetch) is handled by the hosted engine, so there is nothing
extra to configure on your side.

In [1]:
import screamingface as sf

# Point the SDK at the hosted engine. These are also the SDK defaults, so this cell is here to
# make the target explicit, not because it is required.
sf.configure(
    engine_url="https://fusion.dev.screamingface.ai",
    scoreboard_url="https://leaderboard.dev.screamingface.ai",
)

## Look at the leaderboard

Leaderboard discovery reads from the public scoreboard and does not need a connection. This is the
board a result would be compared against. Both calls render as interactive widgets.

In [2]:
sf.leaderboards.list()

Leaderboards(3)

In [3]:
sf.leaderboards.get("draco", top=10)

LeaderboardError: Leaderboard 'draco' is not registered

## Connect a provider

`sf.connect()` opens the connection panel. On the hosted engine it asks you to sign in through
Cloudflare Access first, then to enter your OpenRouter key. The key is sent to the engine, not
kept in the notebook.

In [4]:
sf.connect()

PanelWidget(children=(HTML(value='<style>\n.sf-ui{\n  --sf-bg:#ffffff;--sf-surface:#f6f6f7;--sf-surface-2:#efe…

## Part 1 · Reproduce DRACO

DRACO is a set of research-quality prompts. Each answer is graded against a rubric by a Judge that
the engine owns, so scoring is consistent across candidates. The question DRACO is built to ask is
whether a Fusion of several models answers better than the best single model.

`draco/lite` is a small version: two pinned cases, ten criteria per case, one Judge pass. It is
useful for seeing the flow end to end. Its score is diagnostic and is not comparable to canonical
DRACO.

### The answer and synthesis policies

Each Model answers under the same instruction. The Fusion's synthesizer merges the panel's answers
into one. These are the reference DRACO prompts.

In [5]:
DRACO_ANSWER_PROMPT = (
    "You are answering a research-quality prompt. Provide a thorough, "
    "well-reasoned answer in prose. Address every aspect the prompt raises. "
    "Use clear structure (headings, bullet lists where appropriate) and cite "
    "specific facts, methodologies, or sources where relevant.\n\n"
    "Do not refuse, abstain, or claim uncertainty unless the question is "
    "genuinely ambiguous — the goal is to demonstrate depth of understanding. "
    "Length: aim for the level of detail the question warrants; brevity that "
    "skips key points will be penalised by the rubric."
)

DRACO_SYNTHESIS_PROMPT = (
    "You are synthesising a single, comprehensive answer to a research-quality "
    "prompt by combining N independent answers from a panel of models. The "
    "downstream grader will score your output against a STRUCTURED RUBRIC of "
    "weighted criteria — your goal is to maximise rubric coverage.\n\n"
    "Procedure:\n"
    "1. Read every panel answer carefully.\n"
    "2. Identify which claims, facts, citations, or arguments each panel member "
    "contributes that the others miss.\n"
    "3. Produce ONE unified prose response that:\n"
    "   - Combines the strongest reasoning from every panel member\n"
    "   - Preserves specific named entities, dates, methodologies, and citations\n"
    "   - Resolves disagreements by favouring the more specific / better-cited claim\n"
    "   - Uses clear structure (headings, lists) where it aids the reader\n"
    "4. Do not introduce new facts that no panel member provided.\n"
    "5. Do not hedge or refuse — the panel collectively has enough material.\n\n"
    "Output: the unified prose answer, no preamble, no JSON wrapper."
)

### Define the Models

Two solo Candidates. Each renders as a card; putting a Model's name on its own line displays it.

In [6]:
DRACO_PARAMS = {"max_tokens": 8192, "temperature": 0.0}
DRACO_PARAMS_NO_TEMPERATURE = {"max_tokens": 8192}

gpt = sf.Model(
    "openrouter/openai/gpt-5.5",
    prompt=DRACO_ANSWER_PROMPT,
    params=DRACO_PARAMS_NO_TEMPERATURE,
)
gemini_flash = sf.Model(
    "openrouter/google/gemini-3-flash-preview",
    prompt=DRACO_ANSWER_PROMPT,
    params=DRACO_PARAMS,
)
gpt

Model('openrouter/openai/gpt-5.5', prompt='You are answering a research-quality prompt. Provide a thorough, well-reasoned answer in prose. Address every aspect the prompt raises. Use clear structure (headings, bullet lists where appropriate) and cite specific facts, methodologies, or sources where relevant.\n\nDo not refuse, abstain, or claim uncertainty unless the question is genuinely ambiguous — the goal is to demonstrate depth of understanding. Length: aim for the level of detail the question warrants; brevity that skips key points will be penalised by the rubric.', params={'max_tokens': 8192})

### Combine them into a Fusion

The Fusion runs both Models in parallel, then a synthesizer merges their answers into one. The
synthesizer here is a third model with its own instruction.

In [7]:
draco_fusion = sf.Fusion(
    [gpt, gemini_flash],
    name="draco-fusion",
    synthesizer=sf.Model(
        "openrouter/anthropic/claude-opus-4.8",
        prompt=DRACO_SYNTHESIS_PROMPT,
        params=DRACO_PARAMS,
    ),
)
draco_fusion

Fusion(['gpt-5.5', 'gemini-3-flash-preview'], name='draco-fusion', synthesizer=Model('openrouter/anthropic/claude-opus-4.8', prompt='You are synthesising a single, comprehensive answer to a research-quality prompt by combining N independent answers from a panel of models. The downstream grader will score your output against a STRUCTURED RUBRIC of weighted criteria — your goal is to maximise rubric coverage.\n\nProcedure:\n1. Read every panel answer carefully.\n2. Identify which claims, facts, citations, or arguments each panel member contributes that the others miss.\n3. Produce ONE unified prose response that:\n   - Combines the strongest reasoning from every panel member\n   - Preserves specific named entities, dates, methodologies, and citations\n   - Resolves disagreements by favouring the more specific / better-cited claim\n   - Uses clear structure (headings, lists) where it aids the reader\n4. Do not introduce new facts that no panel member provided.\n5. Do not hedge or refuse — the panel collectively has enough material.\n\nOutput: the unified prose answer, no preamble, no JSON wrapper.', params={'max_tokens': 8192, 'temperature': 0.0}))

### Run the evaluation

One call runs all three Candidates against `draco/lite`. This spends money: two solo answers, one
Fusion (three model calls plus the synthesis), and the Judge grades for two cases. Set
`RUN_EVALUATION = True` to run it. While it runs, a live panel shows progress, calls, tokens, and
cost.

In [8]:
RUN_EVALUATION = True

candidates = [gpt, gemini_flash, draco_fusion]
report = sf.evaluate(candidates, benchmark="draco/lite") if RUN_EVALUATION else None
report if report is not None else "Evaluation disabled — set RUN_EVALUATION = True to spend."

'Evaluation disabled — set RUN_EVALUATION = True to spend.'

### Read the Report

The Report renders as a widget with each Candidate's score, coverage, cost, and the Judge's
per-criterion reasoning. The cells after it pull the same numbers out as plain values.

In [ ]:
report

In [ ]:
if report is not None:
    comparison = sorted(
        (
            {
                "name": result.name,
                "kind": result.kind,
                "score": result.score,
                "coverage": result.coverage,
                "cost_usd": result.usage.cost_usd,
                "failures": len(result.failures),
            }
            for result in report.candidates
        ),
        key=lambda row: (row["score"] is not None, row["score"] or 0.0),
        reverse=True,
    )
else:
    comparison = []
comparison

In [ ]:
if report is not None:
    top = report.candidates[0]
    first_case = top.cases[0]
    case_detail = {
        "candidate": top.name,
        "compiled_url4": top.url4,
        "case_id": first_case.case_id,
        "finish_reason": first_case.finish_reason,
        "grade": first_case.grade,
        "checks": () if first_case.grade is None else first_case.grade.checks,
        "failures": first_case.failures,
    }
else:
    case_detail = None
case_detail

In [ ]:
report.export() if report is not None else "Run the evaluation first to write report.json."

## Part 2 · Edit a run's URL and run it again

Every Candidate you run has a URL4 — a compiled address for the exact expression that ran,
including the benchmark it was linked to. You can read that address, turn it back into editable
Python, change something, and run the new version.

### Read the URL

This is the Fusion you ran in Part 1. A URL4 is a string, so you can display it and read it
directly.

In [ ]:
url4 = report.candidates["draco-fusion"].url4 if report is not None else None
url4 if url4 is not None else "Run Part 1 with RUN_EVALUATION = True to get a URL."

### Turn it into Python

`.to_python()` reconstructs the recipe as editable ScreamingFace code, followed by the evaluation
call that reruns it. Because this URL4 is linked to `draco/lite`, the benchmark comes back with it.

In [ ]:
if url4 is not None:
    print(url4.to_python())
else:
    print("Run Part 1 with RUN_EVALUATION = True to generate the Python fork.")

### Change one model and run it again

Take the code printed above, change one model, and run the new version. The cell below is a
ready-made example of that edit: the same Fusion with the Gemini Flash member replaced by
DeepSeek. Edit it however you like — swap a member, change the synthesizer, or adjust a
parameter.

In [ ]:
edited_candidate = sf.Fusion(
    [
        sf.Model(
            "openrouter/openai/gpt-5.5",
            prompt=DRACO_ANSWER_PROMPT,
            params=DRACO_PARAMS_NO_TEMPERATURE,
        ),
        sf.Model(
            "openrouter/deepseek/deepseek-v4-pro",
            prompt=DRACO_ANSWER_PROMPT,
            params=DRACO_PARAMS,
        ),
    ],
    name="draco-fusion-edited",
    synthesizer=sf.Model(
        "openrouter/anthropic/claude-opus-4.8",
        prompt=DRACO_SYNTHESIS_PROMPT,
        params=DRACO_PARAMS,
    ),
)
edited_candidate

### Run the edited version

This is a fresh, paid evaluation against the same benchmark, so it has its own switch. A fresh run
makes new model calls, so the answers and the score can differ from Part 1.

In [ ]:
RERUN_EVALUATION = False

edited_report = (
    sf.evaluate(edited_candidate, benchmark="draco/lite") if RERUN_EVALUATION else None
)
edited_report if edited_report is not None else "Set RERUN_EVALUATION = True to run the edit."

In [ ]:
if report is not None and edited_report is not None:
    scores = {
        "draco-fusion": report.candidates["draco-fusion"].score,
        "draco-fusion-edited": edited_report.candidates["draco-fusion-edited"].score,
    }
else:
    scores = "Run both evaluations to compare scores."
scores

### Replaying the original unchanged

To rerun the original expression exactly as it was, pass the URL4 straight to `sf.evaluate(url4)`
with no `benchmark` or `limit` — the benchmark is already part of the address.

## Where to go next

- `01_client_tour.ipynb` covers the full client surface.
- `06_draco_full_e2e.ipynb` runs the complete DRACO lineup.
- `07_ifeval_e2e.ipynb` and `08_healthbench_worst30.ipynb` show other benchmarks.
- To put a result on the public leaderboard, see the publish step in `00_quickstart.ipynb`.

As a reminder, every evaluation here spends real money on model calls; the switches keep that
deliberate.